##### Tool Calls

- Declare Tools or Functions in python json format (or python dict)
- Pass them in chat completions function

- The llm responds with finish_reason as tool_calls, it also provides name of tool to call and arguments
- Call the function based on response from llm
- Append the earlier llm response and tool call result (as message) to the list of messages. This creates a proper sequence of messages.

- Now call the llm again, and it will be able to access the response of tool call.

In [1]:
def get_city_temperature(city: str):

    if city.lower() == "bangalore":
        temperature = 25
    else:
        temperature = 50

    return temperature

In [2]:
get_city_temperature_json = {
    "name": "get_city_temperature",
    "description": "Use this tool to get the current temperature of any city in the world",
    "parameters": {
        "type": "object",
        "properties": {
            "city": {
                "type": "string",
                "description": "The name of the city"
            }
        },
        "required": ["city"],
        "additionalProperties": False
    }
}

In [3]:
tools = [{"type": "function", "function": get_city_temperature_json}]

In [4]:
# make openai call, and pass the tools

import os
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()
OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")

client = OpenAI(api_key=OPENAI_API_KEY)

In [20]:
from pprint import pprint

model = "gpt-4o-mini"
messages = [{"role": "user", "content": "What is the temperature of Delhi?"}]

response = client.chat.completions.create(messages=messages, model=model, tools=tools)

pprint(response.__dict__)

{'_request_id': 'req_1746c3e68df144f7940f9822a5283edf',
 'choices': [Choice(finish_reason='tool_calls', index=0, logprobs=None, message=ChatCompletionMessage(content=None, refusal=None, role='assistant', annotations=[], audio=None, function_call=None, tool_calls=[ChatCompletionMessageFunctionToolCall(id='call_QgPPoHOjgeu8tNe4EArdru8o', function=Function(arguments='{"city":"Delhi"}', name='get_city_temperature'), type='function')]))],
 'created': 1772191683,
 'id': 'chatcmpl-DDqIlGoz0bu011Pc5zOk89kf7XkCg',
 'model': 'gpt-4o-mini-2024-07-18',
 'object': 'chat.completion',
 'service_tier': 'default',
 'system_fingerprint': 'fp_0773368f32',
 'usage': CompletionUsage(completion_tokens=15, prompt_tokens=67, total_tokens=82, completion_tokens_details=CompletionTokensDetails(accepted_prediction_tokens=0, audio_tokens=0, reasoning_tokens=0, rejected_prediction_tokens=0), prompt_tokens_details=PromptTokensDetails(audio_tokens=0, cached_tokens=0))}


##### Call the function and provide the outputs back to the llm

In [21]:
import json

message = response.choices[0].message
tool_calls = message.tool_calls

tool_call = tool_calls[0]

function_name = tool_call.function.name
arguments = json.loads(tool_call.function.arguments)

tool = globals().get(function_name)
result = tool(**arguments) if tool else {}

result_message = {"role": "tool","content": json.dumps(result),"tool_call_id": tool_call.id}

print(result_message)

{'role': 'tool', 'content': '50', 'tool_call_id': 'call_QgPPoHOjgeu8tNe4EArdru8o'}


In [22]:
messages.append(message)
messages.append(result_message)

response = client.chat.completions.create(messages=messages, model=model, tools=tools)
pprint(response.__dict__)

{'_request_id': 'req_4dffe19eeb9446e198cd7aee970c02e0',
 'choices': [Choice(finish_reason='stop', index=0, logprobs=None, message=ChatCompletionMessage(content='The current temperature in Delhi is 50°C.', refusal=None, role='assistant', annotations=[], audio=None, function_call=None, tool_calls=None))],
 'created': 1772191697,
 'id': 'chatcmpl-DDqIzvwxzWNpcXtWjHxa0ELJrTV9n',
 'model': 'gpt-4o-mini-2024-07-18',
 'object': 'chat.completion',
 'service_tier': 'default',
 'system_fingerprint': 'fp_0773368f32',
 'usage': CompletionUsage(completion_tokens=11, prompt_tokens=92, total_tokens=103, completion_tokens_details=CompletionTokensDetails(accepted_prediction_tokens=0, audio_tokens=0, reasoning_tokens=0, rejected_prediction_tokens=0), prompt_tokens_details=PromptTokensDetails(audio_tokens=0, cached_tokens=0))}
